# MDM DJBC - Combined Notebook for Google Colab
This notebook contains the code from `Source/step01_setup.py`, `Source/step02_reference.py`, `Source/step03_generators.py`, `Source/step04_simulation.py`, and `Source/step05_profiling.py` assembled so it can be opened and run in Google Colab.

Instructions:
- Run the cells in order.
- The simulation writes CSVs to `data/raw/` and the profiling saves PNGs to `reports/`.

In [ ]:
# === Cell: step01_setup.py (install + imports) ===
# Install optional libraries (works in Colab). Uncomment if needed.
# !pip install missingno faker -q

import pandas as pd
import numpy as np
import re
import random
import warnings
warnings.filterwarnings('ignore')

from faker import Faker
from faker.providers import person, address, company, internet, phone_number

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import missingno as msno

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.width', 120)

sns.set_theme(style='whitegrid', palette='Blues_d')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.family']    = 'sans-serif'

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print('✅ Setup: imports and environment ready')

✅ Setup: imports and environment ready


In [ ]:
# === Cell: step02_reference.py (reference data) ===
PROVINSI = {
    '11': 'Aceh', '12': 'Sumatera Utara', '13': 'Sumatera Barat', '14': 'Riau',
    '15': 'Jambi', '16': 'Sumatera Selatan', '17': 'Bengkulu', '18': 'Lampung',
    '19': 'Kep. Bangka Belitung','21': 'Kep. Riau', '31': 'DKI Jakarta', '32': 'Jawa Barat',
    '33': 'Jawa Tengah', '34': 'DI Yogyakarta', '35': 'Jawa Timur', '36': 'Banten',
    '51': 'Bali', '52': 'Nusa Tenggara Barat', '53': 'Nusa Tenggara Timur', '61': 'Kalimantan Barat',
    '62': 'Kalimantan Tengah', '63': 'Kalimantan Selatan', '64': 'Kalimantan Timur', '65': 'Kalimantan Utara',
    '71': 'Sulawesi Utara', '72': 'Sulawesi Tengah', '73': 'Sulawesi Selatan', '74': 'Sulawesi Tenggara',
    '75': 'Gorontalo', '76': 'Sulawesi Barat', '81': 'Maluku', '82': 'Maluku Utara',
    '91': 'Papua Barat', '94': 'Papua',
}

KPPBC_LIST = [
    {'kode': '010100', 'nama': 'KPU Bea dan Cukai Tipe A Tanjung Priok', 'wilayah': '31'},
    {'kode': '040300', 'nama': 'KPPBC TMP B Soekarno-Hatta',           'wilayah': '31'},
    {'kode': '020300', 'nama': 'KPPBC TMP Belawan',                   'wilayah': '12'},
    {'kode': '070100', 'nama': 'KPPBC TMP Tanjung Perak',             'wilayah': '35'},
    {'kode': '050100', 'nama': 'KPPBC TMP Tanjung Emas',              'wilayah': '33'},
    {'kode': '140100', 'nama': 'KPPBC TMP B Makassar',                'wilayah': '73'},
    {'kode': '060100', 'nama': 'KPPBC TMP A Pasuruan',                'wilayah': '35'},
    {'kode': '090100', 'nama': 'KPPBC TMP B Ngurah Rai',              'wilayah': '51'},
    {'kode': '010700', 'nama': 'KPPBC Tipe Madya Pabean Belawan',     'wilayah': '12'},
    {'kode': '010800', 'nama': 'KPPBC Tipe Madya Pabean B Medan',     'wilayah': '12'},
    {'kode': '011200', 'nama': 'KPPBC Tipe Madya Pabean C Kuala Tanjung', 'wilayah': '12'},
    {'kode': '020400', 'nama': 'KPU Bea dan Cukai Tipe B Batam',      'wilayah': '21'},
    {'kode': '020100', 'nama': 'KPPBC Tipe Madya Pabean B Tanjung Balai Karimun', 'wilayah': '21'},
    {'kode': '030100', 'nama': 'KPPBC Tipe Madya Pabean B Palembang', 'wilayah': '16'},
    {'kode': '030700', 'nama': 'KPPBC Tipe Madya Pabean B Bandar Lampung', 'wilayah': '18'},
    {'kode': '050400', 'nama': 'KPPBC Tipe Madya Pabean Merak',       'wilayah': '36'},
    {'kode': '050900', 'nama': 'KPPBC Tipe Madya Pabean A Bekasi',    'wilayah': '32'},
    {'kode': '070500', 'nama': 'KPPBC TMP Juanda',                    'wilayah': '35'},
    {'kode': '080100', 'nama': 'KPPBC TMP Ngurah Rai',                'wilayah': '51'},
    {'kode': '100300', 'nama': 'KPPBC Balikpapan',                    'wilayah': '64'},
    {'kode': '110100', 'nama': 'KPPBC Makassar',                      'wilayah': '73'},
    {'kode': '120300', 'nama': 'KPPBC Sorong',                        'wilayah': '91'},
    {'kode': '040400', 'nama': 'KPPBC Tipe Madya Pabean A Jakarta',   'wilayah': '31'},
    {'kode': '060300', 'nama': 'KPPBC Tipe Madya Cukai Kudus',        'wilayah': '33'},
    {'kode': '071300', 'nama': 'KPPBC Pasuruan',                      'wilayah': '35'},
    {'kode': '090400', 'nama': 'KPPBC Pontianak',                     'wilayah': '61'},
]

STATUS_NIB_POOL = ['AKTIF', 'DIBEKUKAN', 'DICABUT']
STATUS_BADAN_HUKUM_POOL = ['Berbadan Hukum', 'Belum Berbadan Hukum']
STATUS_PERSEROAN_POOL = ['Aktif', 'Tidak Aktif', 'Dibekukan']
JENIS_API_POOL = ['API-U', 'API-P']
KATEGORI_CEISA_POOL = ['IMPORTIR', 'EKSPORTIR', 'KEDUA-DUANYA']
JENIS_PERSEROAN_POOL = ['PT', 'CV', 'Firma', 'Perum', 'UD']
FLAG_POOL = ['Y', 'N']

print('✅ Reference constants loaded')

In [ ]:
# === Cell: step03_generators.py (generator functions) ===
import random
from datetime import datetime, timedelta

def generate_nib_valid():
    return ''.join([str(random.randint(0, 9)) for _ in range(13)])

def generate_nib_invalid():
    error_types = [
        lambda: ''.join([str(random.randint(0, 9)) for _ in range(12)]),
        lambda: ''.join([str(random.randint(0, 9)) for _ in range(13)]) + 'X',
        lambda: '0000000000000'
    ]
    return random.choice(error_types)()

def generate_npwp_valid():
    d = [str(random.randint(0,99)).zfill(2), str(random.randint(0,999)).zfill(3),
         str(random.randint(0,999)).zfill(3), str(random.randint(0,9)),
         str(random.randint(0,999)).zfill(3), str(random.randint(0,999)).zfill(3)]
    return f"{d[0]}.{d[1]}.{d[2]}.{d[3]}-{d[4]}.{d[5]}"


def generate_npwp_invalid():
    # Parts for generating structurally invalid NPWP
    d = [str(random.randint(0,99)).zfill(2), str(random.randint(0,999)).zfill(3),
         str(random.randint(0,999)).zfill(3), str(random.randint(0,9)),
         str(random.randint(0,999)).zfill(3), str(random.randint(0,999)).zfill(3)]

    raw_15 = ''.join([str(random.randint(0, 9)) for _ in range(15)])

    error_types = [
        lambda: raw_15, # No separators
        lambda: f"{d[0]}{d[1]}.{d[2]}.{d[3]}-{d[4]}.{d[5]}", # Missing first dot
        lambda: raw_15.replace('0', 'O').replace('1', 'I'), # Contains letters
        lambda: f"{d[0]}.{d[1]}{d[2]}.{d[3]}-{d[4]}.{d[5]}", # Missing second dot
        lambda: f"{raw_15[:14]}", # Too short
        lambda: f"{raw_15}X" # Too long
    ]
    return random.choice(error_types)()

def generate_api_valid():
    return ''.join([str(random.randint(0, 9)) for _ in range(10)])

def generate_niper_valid():
    return ''.join([str(random.randint(0, 9)) for _ in range(10)])

def generate_date_random(start_year=2020, end_year=2025):
    start_date = datetime(start_year, 1, 1)
    end_date = datetime(end_year, 12, 31)
    time_between_dates = end_date - start_date
    days_between_dates = time_between_dates.days
    random_days = random.randrange(days_between_dates)
    return (start_date + timedelta(days=random_days)).strftime('%Y-%m-%d')

def generate_date_recent(max_days_ago=90):
    days_ago = random.randint(0, max_days_ago)
    return (datetime.now() - timedelta(days=days_ago)).strftime('%Y-%m-%d')

print('✅ Generator functions ready')

In [ ]:
# === Cell: step04_simulation.py (simulation engine) ===
import pandas as pd
import numpy as np
import random
from faker import Faker
from datetime import datetime

# rely on constants and generator functions defined above
fake = Faker('id_ID')
Faker.seed(42)
random.seed(42)

N_MASTER = 5000

def run_full_simulation():
    print(f"🚀 Memulai simulasi {N_MASTER} perusahaan (DIRTY MODE)... ")

    master_list = []
    for i in range(N_MASTER):
        if random.random() < 0.15:
            nib = generate_nib_invalid()
        else:
            nib = generate_nib_valid()

        npwp = generate_npwp_valid()
        nama = fake.company().upper()
        kppbc = random.choice(KPPBC_LIST)

        flag_impor = random.choice(['Y', 'N'])
        jenis_api = random.choice(JENIS_API_POOL) if flag_impor == 'Y' else ""

        master_list.append({
            'NIB': nib,
            'NPWP_PERSEROAN': npwp,
            'NAMA_PERSEROAN': nama,
            'NAMA_SINGKATAN': nama.split()[0][:5],
            'JENIS_PERSEROAN': random.choice(JENIS_PERSEROAN_POOL),
            'STATUS_BADAN_HUKUM': random.choice(STATUS_BADAN_HUKUM_POOL),
            'STATUS_PERSEROAN': random.choice(STATUS_PERSEROAN_POOL),
            'ALAMAT_PERSEROAN': fake.street_address().upper(),
            'KELURAHAN_PERSEROAN': fake.city().upper(),
            'PERSEROAN_DAERAH_ID': kppbc['wilayah'],
            'KODE_POS_PERSEROAN': fake.postcode(),
            'FLAG_IMPOR': flag_impor,
            'FLAG_EKSPOR': random.choice(['Y', 'N']),
            'JENIS_API': jenis_api,
            'TGL_PERUBAHAN_NIB': generate_date_random(2023, 2025),
            'STATUS_NIB': random.choices(STATUS_NIB_POOL, weights=[80, 10, 10])[0],
            'KODE_KANTOR': kppbc['kode'],
        })

    df_oss = pd.DataFrame(master_list)

    for col in ['KELURAHAN_PERSEROHA N', 'KODE_POS_PERSEROAN', 'NAMA_SINGKATAN']:
        # small safety: if column name typo or not present, skip
        if col in df_oss.columns:
            idx_null = df_oss.sample(frac=0.15).index
            df_oss.loc[idx_null, col] = np.nan

    idx_stale_date = df_oss.sample(frac=0.03).index
    df_oss.loc[idx_stale_date, 'TGL_PERUBAHAN_NIB'] = [generate_date_random(2014, 2015) for _ in range(len(idx_stale_date))]

    df_dups = df_oss.sample(frac=0.10)
    df_oss = pd.concat([df_oss, df_dups], ignore_index=True)

    df_ceisa = df_oss.sample(frac=0.9).copy()

    df_ceisa = df_ceisa.rename(columns={
        'NPWP_PERSEROAN': 'NPWP',
        'NAMA_PERSEROAN': 'NAMA_PERUSAHAAN',
        'ALAMAT_PERSEROAN': 'ALAMAT_PERUSAHAAN',
        'KELURAHAN_PERSEROAN': 'KELURAHAN',
        'PERSEROAN_DAERAH_ID': 'DAERAH_ID',
        'KODE_POS_PERSEROAN': 'KODE_POS',
        'TGL_PERUBAHAN_NIB': 'TGL_TERBIT_NIB'
    })

    df_ceisa['ID_PERUSAHAAN'] = [f"C{str(i).zfill(6)}" for i in range(len(df_ceisa))]
    df_ceisa['NOMOR_TELPON'] = [fake.phone_number() for _ in range(len(df_ceisa))]
    df_ceisa['KATEGORI'] = random.choices(KATEGORI_CEISA_POOL, k=len(df_ceisa))
    df_ceisa['NIPER'] = [generate_niper_valid() if k != 'IMPORTIR' else "" for k in df_ceisa['KATEGORI']]
    df_ceisa['NOMOR_API'] = [generate_api_valid() if k != 'EKSPORTIR' else "" for k in df_ceisa['KATEGORI']]
    df_ceisa['TGL_SYNC_OSS'] = [generate_date_recent(90) for _ in range(len(df_ceisa))]

    ceisa_cols = ['ID_PERUSAHAAN', 'NIB', 'NPWP', 'NAMA_PERUSAHAAN', 'ALAMAT_PERUSAHAAN',
                  'KELURAHAN', 'DAERAH_ID', 'KODE_POS', 'NOMOR_TELPON', 'KATEGORI',
                  'NIPER', 'NOMOR_API', 'TGL_TERBIT_NIB', 'STATUS_NIB', 'KODE_KANTOR', 'TGL_SYNC_OSS']
    df_ceisa = df_ceisa[ceisa_cols]

    # Inject anomalies (kept simple)
    idx_npwp = df_ceisa.sample(frac=0.25).index
    df_ceisa.loc[idx_npwp, 'NPWP'] = [generate_npwp_invalid() for _ in range(len(idx_npwp))]

    idx_fuzzy = df_ceisa.sample(frac=0.15).index
    df_ceisa.loc[idx_fuzzy, 'NAMA_PERUSAHAAN'] = df_ceisa.loc[idx_fuzzy, 'NAMA_PERUSAHAAN'].apply(lambda x: str(x).replace("PT ", "") + " (CABANG)")

    idx_logic = df_ceisa.sample(frac=0.15).index
    df_ceisa.loc[idx_logic, 'NIPER'] = "4803163678"
    df_oss.loc[df_oss.index.isin(idx_logic), 'FLAG_EKSPOR'] = 'N'

    idx_conflict = df_ceisa.sample(n=200).index
    df_ceisa.loc[idx_conflict, 'STATUS_NIB'] = 'AKTIF'
    df_oss.loc[df_oss.index.isin(idx_conflict), 'STATUS_NIB'] = 'DICABUT'

    idx_snapshot = df_ceisa.sample(frac=0.05).index
    df_snapshot_old = df_ceisa.loc[idx_snapshot].copy()
    df_snapshot_old['NAMA_PERUSAHAAN'] = df_snapshot_old['NAMA_PERUSAHAAN'] + ' (OLD)'
    df_snapshot_old['TGL_SYNC_OSS'] = [generate_date_random(2023, 2024) for _ in range(len(df_snapshot_old))]
    df_ceisa = pd.concat([df_ceisa, df_snapshot_old], ignore_index=True)

    # Ensure data directories exist
    import os
    os.makedirs('data/raw', exist_ok=True)
    os.makedirs('reports', exist_ok=True)

    df_oss = df_oss.drop(columns=['KODE_KANTOR'], errors='ignore')
    df_oss.to_csv('data/raw/oss_nib_data.csv', index=False)
    df_ceisa.to_csv('data/raw/ceisa_data.csv', index=False)

    print(f"✅ Simulasi Berhasil (DIRTY DATA READY)!")
    print(f"   - OSS: {len(df_oss)} records")
    print(f"   - CEISA: {len(df_ceisa)} records")

if __name__ == '__main__':
    run_full_simulation()

In [ ]:
# === Cell: step05_profiling.py (DQ metrics & visualization) ===
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from datetime import datetime

def calculate_dq_metrics(df, dataset_name="OSS"):
    n_rows = len(df)
    results = {}

    mand_cols = ['NIB', 'STATUS_NIB']
    mand_cols += ['NPWP_PERSEROAN', 'NAMA_PERSEROAN'] if dataset_name == 'OSS' else ['NPWP', 'NAMA_PERUSAHAAN']
    comp_score = (1 - df[mand_cols].isnull().any(axis=1).sum() / n_rows) * 100
    results['Kelengkapan'] = comp_score

    v_nib = df['NIB'].astype(str).str.match(r'^\d{13}$').mean() * 100
    npwp_col = 'NPWP_PERSEROAN' if dataset_name == 'OSS' else 'NPWP'
    v_npwp = df[npwp_col].astype(str).str.match(r'^\d{2}\.\d{3}\.\d{3}\.\d{1}-\d{3}\.\d{3}$').mean() * 100
    results['Validitas'] = (v_nib + v_npwp) / 2

    uniq_score = (df['NIB'].nunique() / n_rows) * 100
    results['Unik'] = uniq_score

    if dataset_name == 'OSS':
        tgl = pd.to_datetime(df['TGL_PERUBAHAN_NIB'], errors='coerce')
        is_stale = (datetime.now() - tgl).dt.days > 365
        timeliness_score = (1 - is_stale.sum() / n_rows) * 100
    else:
        tgl_sync = pd.to_datetime(df['TGL_SYNC_OSS'], errors='coerce')
        high_sync_lag = (datetime.now() - tgl_sync).dt.days > 30
        timeliness_score = (1 - high_sync_lag.sum() / n_rows) * 100
    results['Ketepatan Waktu'] = timeliness_score

    return results

def generate_professional_scorecard(metrics, title_suffix="CEISA"):
    dim_names = list(metrics.keys())
    dim_scores = list(metrics.values())
    total_dq_score = np.mean(dim_scores)

    TARGET_SCORE = 80
    WARNING_SCORE = 70

    if total_dq_score >= TARGET_SCORE: grade, grade_color = 'A (Excellent)', '#4CAF50'
    elif total_dq_score >= WARNING_SCORE: grade, grade_color = 'B (Good)', '#8BC34A'
    elif total_dq_score >= 60: grade, grade_color = 'C (Fair)', '#FF9800'
    else: grade, grade_color = 'D (Poor)', '#F44336'

    fig = plt.figure(figsize=(16, 8))
    fig.suptitle(f'Data Quality Scorecard — {title_suffix} (Before MDM)', fontsize=16, fontweight='bold', y=1.05)

    angles = np.linspace(0, 2*np.pi, len(dim_names), endpoint=False).tolist()
    dim_scores_polar = dim_scores + [dim_scores[0]]
    angles += [angles[0]]

    ax1 = plt.subplot(121, polar=True)
    ax1.set_theta_offset(np.pi / 2)
    ax1.set_theta_direction(-1)

    ax1.plot(angles, dim_scores_polar, 'o-', linewidth=3, color='#2196F3', markersize=8)
    ax1.fill(angles, dim_scores_polar, alpha=0.3, color='#2196F3')
    ax1.plot(angles, [TARGET_SCORE]*len(angles), '--', color='red', alpha=0.6, label=f'Target {TARGET_SCORE}%')

    ax1.set_xticks(angles[:-1])
    ax1.set_xticklabels(dim_names, fontsize=10, fontweight='bold')
    ax1.set_ylim(0, 100)
    ax1.set_yticks([20, 40, 60, 80, 100])
    ax1.set_yticklabels(['20', '40', '60', '80', '100%'], fontsize=8)
    ax1.set_title('DQ Score Dimensions (DMBOK)', fontweight='bold', pad=30)
    ax1.legend(loc='upper right', bbox_to_anchor=(0.1, 0.1), fontsize=9)

    for angle, score in zip(angles[:-1], dim_scores):
        ax1.annotate(f'{score:.1f}%', xy=(angle, score), fontsize=9, ha='center',
                     xytext=(0, 10), textcoords='offset points', color='white', 
                     fontweight='bold', bbox=dict(boxstyle='round,pad=0.3', fc='#1565C0', alpha=0.8))

    ax2 = plt.subplot(122)
    y_pos = np.arange(len(dim_names))
    colors2 = [grade_color if s >= TARGET_SCORE else '#FF9800' if s >= WARNING_SCORE else '#F44336' for s in dim_scores]

    bars2 = ax2.barh(y_pos, dim_scores, color=colors2, edgecolor='white', height=0.6)
    ax2.axvline(x=TARGET_SCORE, color='red', linestyle='--', linewidth=2, label=f'Target {TARGET_SCORE}%')

    ax2.set_yticks(y_pos)
    ax2.set_yticklabels(dim_names, fontsize=10, fontweight='bold')
    ax2.set_xlim(0, 105)
    ax2.set_xlabel('Score (%)')
    ax2.set_title((f'DQ Score vs Target\n'
                   f'(Avg Score: {total_dq_score:.1f}/100 | Grade: {grade})'), fontweight='bold')

    for bar, score in zip(bars2, dim_scores):
        ax2.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2, f'{score:.1f}%', va='center', fontweight='bold', fontsize=10)

    plt.tight_layout()
    filename = f'reports/dq_scorecard_{title_suffix.lower().replace(' ', '_')}.png'
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    print(f'💾 Chart disimpan: {filename}')
    plt.show()

print('✅ Profiling utilities ready')


In [ ]:
# === Cell: Run simulation + profiling (execute after previous cells) ===
# Run the simulation (may take some time)
run_full_simulation()

# Load generated files and produce scorecards
import pandas as pd
try:
    df_oss = pd.read_csv('data/raw/oss_nib_data.csv')
    df_ceisa = pd.read_csv('data/raw/ceisa_data.csv')

    print(f'\n📊 Generating DQ Scorecard for OSS Master...')
    oss_metrics = calculate_dq_metrics(df_oss, 'OSS')
    generate_professional_scorecard(oss_metrics, 'OSS Master')

    print(f'\n📊 Generating DQ Scorecard for CEISA Operational...')
    ceisa_metrics = calculate_dq_metrics(df_ceisa, 'CEISA')
    generate_professional_scorecard(ceisa_metrics, 'CEISA Operational')

    print(f'\n' + '='*45)
    print('   FINAL BASELINE DMBOK SUMMARY (4-DIM)')
    print('='*45)
    import numpy as np
    print(f"OSS Master Overall Score      : {np.mean(list(oss_metrics.values())):.1f}/100")
    print(f"CEISA Operational Overall Score: {np.mean(list(ceisa_metrics.values())):.1f}/100")
    print('='*45)
except Exception as e:
    print(f'❌ Error while generating reports: {e}')